In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df.drop(columns=['customerID'], inplace=True)
print("Shape:", df.shape)
print("\nNaN count:", df.isnull().sum().sum())
print("\nObject columns:")
print(df.select_dtypes(include='object').columns.tolist())
print("\nSample:")
print(df.head(3))

Shape: (7043, 20)

NaN count: 0

Object columns:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']

Sample:
   gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  Female              0     Yes         No       1           No   
1    Male              0      No         No      34          Yes   
2    Male              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity OnlineBackup  \
0  No phone service             DSL             No          Yes   
1                No             DSL            Yes           No   
2                No             DSL            Yes          Yes   

  DeviceProtection TechSupport StreamingTV StreamingMovies        Contract  \
0               No          No          No              No  Month-to-mont

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18284\2845507324.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


In [11]:
("Churn unique values before encoding:", df['Churn'].unique())
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print("Churn unique values after encoding:", df['Churn'].unique())
print("Churn value counts:\n", df['Churn'].value_counts())

df['gender'] = (df['gender'] == 'Female').astype(int)
df['Partner'] = (df['Partner'] == 'Yes').astype(int)
df['Dependents'] = (df['Dependents'] == 'Yes').astype(int)
df['PhoneService'] = (df['PhoneService'] == 'Yes').astype(int)
df['PaperlessBilling'] = (df['PaperlessBilling'] == 'Yes').astype(int)

service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
df['NumServices'] = df[service_cols].apply(
    lambda row: (row == 'Yes').sum(), axis=1)
df['AvgMonthlySpend'] = df['TotalCharges'] / (df['tenure'] + 1)
df['IsMonthToMonth'] = (df['Contract'] == 'Month-to-month').astype(int)
df['ChargePerService'] = df['MonthlyCharges'] / (df['NumServices'] + 1)
df['IsHighValue'] = (df['MonthlyCharges'] > 
                     df['MonthlyCharges'].median()).astype(int)
df['TenureGroup'] = pd.cut(df['tenure'],
                            bins=[0, 12, 36, 72],
                            labels=[0, 1, 2],
                            include_lowest=True).astype(float).fillna(0).astype(int)
multi_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity',
              'OnlineBackup', 'DeviceProtection', 'TechSupport',
              'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']
df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

for col in df.select_dtypes(include='bool').columns:
    df[col] = df[col].astype(int)

print("Shape:", df.shape)
print("NaNs:", df.isnull().sum().sum())
print("Object cols:", df.select_dtypes(include='object').columns.tolist())

Churn unique values after encoding: [0 1]
Churn value counts:
 Churn
0    5174
1    1869
Name: count, dtype: int64
Shape: (7043, 37)
NaNs: 0
Object cols: []


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges',
            'AvgMonthlySpend', 'ChargePerService']
X_train = X_train.copy()
X_test = X_test.copy()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("NaNs in train:", X_train.isnull().sum().sum())
print("NaNs in test:", X_test.isnull().sum().sum())

Train: (5634, 36)
Test: (1409, 36)
NaNs in train: 0
NaNs in test: 0


In [13]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

mlflow.set_experiment("churn_prediction")
print("MLflow ready!")
print("Tracking URI:", mlflow.get_tracking_uri())

MLflow ready!
Tracking URI: sqlite:///C:/Users/Lenovo/churn-prediction-mlops/notebooks/mlflow.db


In [14]:
print("y_train unique values:", y_train.unique())
print("y_test unique values:", y_test.unique())
print("y_train value counts:\n", y_train.value_counts())

y_train unique values: [0 1]
y_test unique values: [0 1]
y_train value counts:
 Churn
0    4139
1    1495
Name: count, dtype: int64


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

with mlflow.start_run(run_name="Logistic_Regression"):
    lr = LogisticRegression(class_weight='balanced',
                             random_state=42, max_iter=1000)
    lr.fit(X_train, y_train)
    y_pred = lr.predict(X_test)
    y_prob = lr.predict_proba(X_test)[:, 1]

    metrics = {
        "auc_roc": roc_auc_score(y_test, y_prob),
        "f1": f1_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred)
    }
    mlflow.log_params({"model": "LogisticRegression",
                        "class_weight": "balanced"})
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(lr, "model")

    print("=== Logistic Regression ===")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

2026/03/19 16:17:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/19 16:17:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


=== Logistic Regression ===
auc_roc: 0.8477
f1: 0.6162
precision: 0.5078
recall: 0.7834


In [18]:
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name="Random_Forest"):
    rf = RandomForestClassifier(n_estimators=100,
                                 class_weight='balanced',
                                 random_state=42)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    y_prob = rf.predict_proba(X_test)[:, 1]

    metrics = {
        "auc_roc": roc_auc_score(y_test, y_prob),
        "f1": f1_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred)
    }
    mlflow.log_params({"model": "RandomForest",
                        "n_estimators": 100})
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(rf, "model")

    print("=== Random Forest ===")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

2026/03/19 16:21:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/19 16:21:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


=== Random Forest ===
auc_roc: 0.8230
f1: 0.5339
precision: 0.6125
recall: 0.4733


In [17]:
from xgboost import XGBClassifier

with mlflow.start_run(run_name="XGBoost"):
    xgb = XGBClassifier(scale_pos_weight=3,
                         n_estimators=100,
                         learning_rate=0.1,
                         random_state=42,
                         eval_metric='logloss')
    xgb.fit(X_train, y_train)
    y_pred = xgb.predict(X_test)
    y_prob = xgb.predict_proba(X_test)[:, 1]

    metrics = {
        "auc_roc": roc_auc_score(y_test, y_prob),
        "f1": f1_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred)
    }
    mlflow.log_params({"model": "XGBoost",
                        "scale_pos_weight": 3,
                        "learning_rate": 0.1})
    mlflow.log_metrics(metrics)
    mlflow.xgboost.log_model(xgb, "model")

    print("=== XGBoost ===")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

2026/03/19 16:20:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


=== XGBoost ===
auc_roc: 0.8411
f1: 0.6262
precision: 0.5264
recall: 0.7727


In [3]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
runs = mlflow.search_runs()
print(runs.columns.tolist())
print(runs)

['run_id', 'experiment_id', 'status', 'artifact_uri', 'start_time', 'end_time']
Empty DataFrame
Columns: [run_id, experiment_id, status, artifact_uri, start_time, end_time]
Index: []


In [4]:
import mlflow

# Set the tracking URI to where runs are saved
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# Get all runs from the experiment
runs = mlflow.search_runs()
print(runs[['run_id', 'tags.mlflow.runName', 
            'metrics.auc_roc', 'metrics.f1', 
            'metrics.precision', 'metrics.recall']])

KeyError: "['tags.mlflow.runName', 'metrics.auc_roc', 'metrics.f1', 'metrics.precision', 'metrics.recall'] not in index"